In [14]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.ensemble import RandomForestRegressor

import joblib
from utils import evaluate_across_gws, select_features_by_correlation, tune_best_model


In [15]:
player_data = pd.read_csv('../data/rolled_data_24_25.csv', index_col=False)
# player_data_23_24 = pd.read_csv('../data/rolled_data_23_24.csv', index_col=False)

TARGET_COL = 'total_points'
OUTPUT_SUFFIX = 'pts'
DROP_COLS = (TARGET_COL, 'round', 'element')

POSITIONS = {
    'gk': 'Goalkeeper',
    'def': 'Defender',
    'mid': 'Midfielder',
    'fwd': 'Forward',
}

AVAIL_FEATS_BY_POS = {
    'gk': [
        'was_home', 'difficulty', 'opponent_difficulty', 'win_prob', 'percentage_net_transfers', 'elo_diff', 'points_5_ewm', 'xP_5_ewm', 'minutes_rolling_3',
        'clean_sheets_3_ewm', 'goals_conceded_3_ewm', 'yellow_cards_rolling_5', 'saves_rolling_5', 'ict_index_rolling_3', 'starts_rolling_3',
        'shots_faced_3_ewm', 'saves_3_ewm', 'ict_index_3_ewm', 'expected_goals_conceded_3_ewm', 'goals_prevented_3_ewm', 'opponent_xG_3_ewm',
        ],

    'def': [
        'was_home', 'difficulty', 'opponent_difficulty', 'win_prob', 'elo_diff', 'yellow_cards_rolling_5', 'starts_rolling_3',  'points_3_ewm', 'xP_3_ewm',
        'minutes_rolling_3', 'goals_scored_3_ewm', 'assists_3_ewm', 'clean_sheets_3_ewm', 'goals_conceded_3_ewm', 'ict_index_3_ewm', 'expected_goals_3_ewm', 'expected_assists_3_ewm',
        'expected_goal_involvements_3_ewm', 'expected_goals_conceded_3_ewm', 'interceptions_3_ewm', 'blocks_3_ewm', 'clearances_3_ewm', 'tackles_3_ewm',
        'shots_faced_3_ewm',  'goals_prevented_3_ewm',
    ],

    'mid': [
        'was_home', 'difficulty', 'opponent_difficulty', 'win_prob', 'elo_diff', 'yellow_cards_rolling_5', 'starts_rolling_3',  'points_3_ewm', 'xP_3_ewm',
        'minutes_rolling_3', 'goals_scored_3_ewm', 'assists_3_ewm', 'clean_sheets_3_ewm', 'goals_conceded_3_ewm', 'ict_index_3_ewm', 'expected_goals_3_ewm', 'expected_assists_3_ewm',
        'expected_goal_involvements_3_ewm', 'expected_goals_conceded_3_ewm', 'interceptions_3_ewm', 'blocks_3_ewm', 'clearances_3_ewm', 'tackles_3_ewm',
        'shots_faced_3_ewm',  'goals_prevented_3_ewm'
    ],

    'fwd': [
        'was_home', 'difficulty', 'opponent_difficulty', 'win_prob', 'elo_diff', 'yellow_cards_rolling_5', 'starts_rolling_3',  'points_3_ewm', 'xP_3_ewm',
        'minutes_rolling_3', 'goals_scored_3_ewm', 'assists_3_ewm',  'ict_index_3_ewm', 'expected_goals_3_ewm', 'expected_assists_3_ewm',
        'expected_goal_involvements_3_ewm', 'interceptions_3_ewm', 'blocks_3_ewm', 'clearances_3_ewm', 'tackles_3_ewm'
    ]
}

## Train + predict, per position


In [16]:
def train_position(suffix, position, blank_gw_rounds=(1,2), n_iter=20,
                    cv_window=10, retune_every=5, verbose=True):
    """ Saves `hist/feats_{suffix}_{OUTPUT_SUFFIX}`,
    `models_by_round_{suffix}_{OUTPUT_SUFFIX}`, and `hist/{suffix}_preds_{OUTPUT_SUFFIX}.csv`;
    returns (preds_df, mae_by_round, r2_by_round) for a quick sanity check.
    """
    data = player_data[player_data['position'] == position]
    # data_23_24 = player_data_23_24[player_data_23_24['position'] == position]
    avail_feats = AVAIL_FEATS_BY_POS[suffix]

    active_mask = data['minutes_rolling_3'].notna() | (data['minutes_rolling_5'] > 0)
    # active_mask_23_24 = data_23_24['minutes_rolling_3'].notna() | (data_23_24['minutes_rolling_5'] > 0)

    data_train_pool = data[active_mask].fillna(0)
    # data_train_pool_23_24 = data_23_24[active_mask_23_24].fillna(0)

    # --- feature selection per round ---
    feats = {}
    for round_ in range(6, 39):
        curr = data_train_pool[(data_train_pool['round'] > 3) & (data_train_pool['round'] < round_)].fillna(0)
        # all_curr =
        feats[round_] = select_features_by_correlation(
            curr, avail_feats, target_col=TARGET_COL, min_corr=0.1,
            redundancy_threshold=0.5, verbose=False,
        )
    joblib.dump(feats, f'./hist/feats_{suffix}_{OUTPUT_SUFFIX}')
    if verbose:
        print(f"[{suffix}] feature selection done for {len(feats)} rounds")

    # --- model comparison + tuning per round (capped window, periodic retuning) ---
    models_by_round = {}
    last_selection = None
    for round_ in range(5, 38):
        feats_round = feats[round_ + 1] + ['round', 'element', 'was_home', TARGET_COL]

        if last_selection is None or (round_ - 5) % retune_every == 0:
            gw_lo = max(3, round_ - cv_window + 1)
            combined, summary = evaluate_across_gws(
                data_train_pool[feats_round], gw_list=range(gw_lo, round_ + 1),
                target_col=TARGET_COL, drop_cols=DROP_COLS,
            )
            base_name = summary.sort_values("MAE").iloc[0]["model"]
            base_mae = summary.sort_values("MAE").iloc[0]["MAE"]
            base_r2 = summary.sort_values("MAE").iloc[0]["R2"]

            val_lo = max(1, round_ - cv_window)
            best_model, best_estimator, search, c_final, mae, rmse, r2 = tune_best_model(
                data_train_pool[feats_round], summary, val_gws=range(val_lo, round_), final_test_gw=round_ + 1,
                target_col=TARGET_COL, drop_cols=DROP_COLS, n_iter=n_iter,
            )

            if mae > base_mae:
                # `search` (the full RandomizedSearchCV object, with per-fold CV results)
                # is deliberately NOT kept here -- nothing downstream reads it, and
                # keeping it around bloated a single position's joblib dump past 100MB
                # for no benefit.
                last_selection = {
                    'mae': mae, 'r2': r2, 'best_model': best_model, 'c_final': c_final,
                    'best_estimator': best_estimator,
                }
            else:
                last_selection = {
                    'mae': base_mae, 'r2': base_r2, 'best_model': base_name, 'c_final': c_final,
                }
            if verbose:
                print(f"[{suffix}] round {round_ + 1}: RETUNED, best={last_selection['best_model']} "
                      f"MAE={last_selection['mae']:.3f}")
        elif verbose:
            print(f"[{suffix}] round {round_ + 1}: reusing last tuned selection "
                  f"({last_selection['best_model']})")

        models_by_round[round_ + 1] = last_selection
    joblib.dump(models_by_round, f'./models_by_round_{suffix}_{OUTPUT_SUFFIX}')

    # --- walk-forward predictions, same structure as get predictions.ipynb ---
    preds_df = pd.DataFrame()
    mae_by_round, r2_by_round = {}, {}
    cached_model = None

    for ro in range(6, 39):
        tar_feats = feats[ro].copy()

        # Train on active players only; test/predict on the FULL population for this
        # round (everyone needs a prediction downstream, whether or not they've
        # recently played).
        X_train = data_train_pool[data_train_pool['round'] < ro][tar_feats]
        y_train = data_train_pool[data_train_pool['round'] < ro][TARGET_COL]
        c = abs(y_train.min()) + 1.0
        y_train_shifted = y_train + c

        X_test = data[data['round'] == ro][tar_feats]
        X_test_index = data[data['round'] == ro]['element']
        X_test_names = data[data['round'] == ro]['name']
        y_test = data[data['round'] == ro][TARGET_COL]

        if ro in blank_gw_rounds:
            # blank-gameweek rounds: not enough fixtures that week to retrain sensibly,
            # so reuse last round's fitted model on this round's (smaller) feature set,
            # same special-casing as the xP predictors.
            model = cached_model
            tar_feats = feats[ro - 1].copy()
            X_test = data[data['round'] == ro][tar_feats]
            X_test_index = data[data['round'] == ro]['element']
            X_test_names = data[data['round'] == ro]['name']
            y_test = data[data['round'] == ro][TARGET_COL]
        else:
            model_info = models_by_round[ro]
            if 'best_estimator' in model_info:
                model = model_info['best_estimator'].regressor  # not fitted yet
            else:
                model = RandomForestRegressor()
            model.fit(X_train, y_train_shifted)

        target_preds = model.predict(X_test) - c

        preds = {
            'element': X_test_index.values, 'name': X_test_names.values,
            f'{TARGET_COL}_pred': target_preds, f'{TARGET_COL}_actual': y_test.values, 'round': ro,
        }
        mae = mean_absolute_error(y_test, target_preds)
        rmse = np.sqrt(mean_squared_error(y_test, target_preds))
        r2 = r2_score(y_test, target_preds)
        mae_by_round[ro], r2_by_round[ro] = mae, r2
        if verbose:
            print(f"[{suffix}] round {ro} - MAE: {mae:.4f}, RMSE: {rmse:.4f}, R2: {r2:.4f}")

        preds_df = pd.concat([preds_df, pd.DataFrame(preds)], ignore_index=True)
        cached_model = model

    preds_df.to_csv(f'./hist/{suffix}_preds_{OUTPUT_SUFFIX}.csv', index=False)
    return preds_df, mae_by_round, r2_by_round


In [ ]:
results = {}
for suffix, position in POSITIONS.items():
    print(f"\n=== {position} ===")
    results[suffix] = train_position(suffix, position)



=== Goalkeeper ===
[gk] feature selection done for 33 rounds
Best model from comparison: Ridge
Fitting 2 folds for each of 20 candidates, totalling 40 fits

Best CV MAE: 0.892
Best params:
  regressor__alpha: 0.042070539502879395

Holdout (round 6) performance:
  MAE=0.458  RMSE=0.894  R2=0.500
[gk] round 6: RETUNED, best=Ridge MAE=0.911
[gk] round 7: reusing last tuned selection (Ridge)
[gk] round 8: reusing last tuned selection (Ridge)
[gk] round 9: reusing last tuned selection (Ridge)
[gk] round 10: reusing last tuned selection (Ridge)


### Sanity check: active-player-filtered R2 vs the original (unfiltered) xP model


In [ ]:
summary_rows = []
for suffix, position in POSITIONS.items():
    preds_df, mae_by_round, r2_by_round = results[suffix]
    # orig_preds = pd.read_csv(f'./hist/{suffix}_preds.csv')  # original xP-targeted predictions, no active-player filter

    new_r2 = r2_score(preds_df[f'{TARGET_COL}_actual'], preds_df[f'{TARGET_COL}_pred'])
    # orig_r2 = r2_score(orig_preds['xP_actual'], orig_preds['xP_pred'])

    summary_rows.append({
        'position': position,
        f'R2 (active-filtered {TARGET_COL} model)': new_r2,
        # 'R2 (original xP model, unfiltered)': orig_r2,
    })

pd.DataFrame(summary_rows)

KeyError: 'fwd'